In [5]:
#  Load All Data 

import pandas as pd
import numpy as np
from pathlib import Path

census = pd.read_parquet("data/processed/census_processed.parquet")
sahie  = pd.read_parquet("data/raw/sahie/sahie_raw.parquet")
cbp    = pd.read_parquet("data/raw/cbp/cbp_raw.parquet")

In [6]:
# Standardize Year Column Names

# Rename year columns to a common name
census_std = census.rename(columns={"acs_year":   "year"})
sahie_std  = sahie.rename(columns={"sahie_year": "year"})
cbp_std    = cbp.rename(columns={"cbp_year":    "year"})

print("Year columns standardized")
print(f"\n  Census year column : 'year'")
print(f"  SAHIE year column  : 'year'")
print(f"  CBP year column    : 'year'")

Year columns standardized

  Census year column : 'year'
  SAHIE year column  : 'year'
  CBP year column    : 'year'


In [7]:
#  Merge All Sources 

# Start with Census as the base (has all years + all geographies)
merged = census_std.copy()

# Select only the value columns from SAHIE 
# (drop duplicate metadata columns)
sahie_values = sahie_std[[
    "geography", "year",
    "total_under_65",
    "num_insured",
    "num_uninsured",
    "pct_uninsured",
    "pct_insured",
]]

# Select only value columns from CBP 
cbp_values = cbp_std[[
    "geography", "year",
    "total_employment",
    "employer_establishments",
]]

#  Merge SAHIE into Census 
merged = merged.merge(
    sahie_values,
    on=["geography", "year"],
    how="left"
)

# Merge CBP into merged 
merged = merged.merge(
    cbp_values,
    on=["geography", "year"],
    how="left"
)

print("All sources merged")
print(f"\nMerged dataset:")
print(f"   Rows    : {len(merged)}")
print(f"   Columns : {len(merged.columns)}")

# Check what's populated for each geography
print(f"\nData coverage:")
for geo in merged["geography"].unique():
    sub = merged[merged["geography"] == geo]
    print(f"\n  {geo}")
    print(f"    Rows with Census data : "
          f"{sub['total_population'].notna().sum()}")
    print(f"    Rows with SAHIE data  : "
          f"{sub['pct_insured'].notna().sum()}")
    print(f"    Rows with CBP data    : "
          f"{sub['total_employment'].notna().sum()}")

All sources merged

Merged dataset:
   Rows    : 45
   Columns : 87

Data coverage:

  California
    Rows with Census data : 15
    Rows with SAHIE data  : 14
    Rows with CBP data    : 0

  Tehama County
    Rows with Census data : 15
    Rows with SAHIE data  : 14
    Rows with CBP data    : 12

  United States
    Rows with Census data : 15
    Rows with SAHIE data  : 14
    Rows with CBP data    : 0


In [9]:
# Save Final Power BI Dataset 

FINAL_DIR = Path("data/BI_Data")
FINAL_DIR.mkdir(parents=True, exist_ok=True)

parquet_path = FINAL_DIR / "tehama_demographic_data.parquet"
csv_path     = FINAL_DIR / "tehama_demographic_data.csv"

merged.to_parquet(parquet_path, index=False)
merged.to_csv(csv_path, index=False)

print(f" Power BI dataset saved")
print(f"\n  Parquet : {parquet_path}")
print(f"\n  CSV     : {csv_path}")

print(f"\n  Rows    : {len(merged)}")
print(f"  Columns : {len(merged.columns)}")
print(f"  Years   : {merged['year'].min()} → {merged['year'].max()}")
print(f"  Geos    : {merged['geography'].unique().tolist()}")

 Power BI dataset saved

  Parquet : data\BI_Data\tehama_demographic_data.parquet

  CSV     : data\BI_Data\tehama_demographic_data.csv

  Rows    : 45
  Columns : 87
  Years   : 2010 → 2024
  Geos    : ['California', 'Tehama County', 'United States']
